# Benchmark Project Throughput

Measure the mm2s -> stream_throughput -> krnl_proj -> s2mm path (no network) to understand the pattern-matching pipeline throughput after the 4-byte DCAM update.

krnl_proj outputs sparse events + one report per packet; output bytes != input bytes.

This xclbin is built via `make all DESIGN=benchmark_project` and includes the helper `stream_throughput`, which samples an AXI stream and reports:
- `cycles`: clock cycles elapsed per sampling window
- `bytes`: bytes transferred in that window
- `ready_not_valid`: cycles receiver was ready but source had no data (should be ~0 with mm2s)
- `valid_not_ready`: cycles source had data but receiver was not ready (backpressure from downstream)

Adjust `DCAM_BYTES_PER_CYCLE` and buffer sizes if you change the scanner throughput.


In [1]:
# Uncomment to reset FPGA if needed
!xbutil reset --device 0000:02:00.1 --force


Performing 'HOT Reset' on '0000:02:00.1'
Are you sure you wish to proceed? [Y/n]: Y (Force override)
Successfully reset Device[0000:02:00.1]


In [2]:
import numpy as np
import pynq
from pathlib import Path


In [3]:
# Select device and xclbin
currentDevice = pynq.Device.devices[0]

# mm2s → throughput → proj → s2mm
# xclbin = Path('/home/m2_1/dat480_project_base/benchmark_project.intf0.xilinx_u55c_gen3x16_xdma_3_202210_1/vnx_benchmark_project_if0.xclbin')
# mm2s → proj → throughput → s2mm
xclbin = Path('/home/m2_1/dat480_project_base/benchmark_project2.intf0.xilinx_u55c_gen3x16_xdma_3_202210_1/vnx_benchmark_project2_if0.xclbin')

if not xclbin.exists():
    raise FileNotFoundError(f'Missing xclbin: {xclbin}')
ol = pynq.Overlay(str(xclbin), device=currentDevice)
print(f"Loaded {xclbin.name} on {currentDevice.name}")
# Sanity check kernel handles exist
_ = (ol.stream_throughput_0, ol.krnl_proj_0, ol.krnl_mm2s_0, ol.krnl_s2mm_0)


Loaded vnx_benchmark_project2_if0.xclbin on xilinx_u55c_gen3x16_xdma_base_3


In [4]:
# %% [markdown]
# ## Module 2 - Sampling config and parameters

# %%
sample_t = np.dtype([
    ('cycles', 'u4'),
    ('bytes', 'u4'),
    ('ready_not_valid', 'u4'),
    ('valid_not_ready', 'u4'),
], align=False)

N_SAMPLES = 100
N_RATE = 1000

# For reference only (your scanner parallelism)
DCAM_BYTES_PER_CYCLE = 4  # update if DCAM changes

# Input sized to cover all sampling windows (stable steady-state)
SIZE_IN = 64 * N_RATE * (N_SAMPLES + 1)

# Estimate TLAST packet count (assume mm2s default packet size = 1408B)
PKT_BYTES = 1408
num_pkts = (SIZE_IN + PKT_BYTES - 1) // PKT_BYTES

# num_pkts = 16

# Output buffer: do NOT use worst-case bound; sink just needs to absorb data during sampling
SIZE_OUT = 8 * 1024 * 1024  # 8MB

print(f'Input bytes: {SIZE_IN:,}  (estimated packets={num_pkts}, pkt_bytes={PKT_BYTES})')
print(f'Output buffer bytes: {SIZE_OUT:,}')
print(f'Sampling: {N_SAMPLES} windows, rate={N_RATE} cycles/window')


Input bytes: 6,464,000  (estimated packets=4591, pkt_bytes=1408)
Output buffer bytes: 8,388,608
Sampling: 100 windows, rate=1000 cycles/window


In [5]:
# %% [markdown]
# ## Module 3 - Allocate buffers and initialize input

# %%
target = ol.HBM0 if hasattr(ol, 'HBM0') else (ol.bank1 if hasattr(ol, 'bank1') else None)
if target is None:
    raise RuntimeError("No suitable memory bank found (HBM0/bank1 missing on this overlay).")

perf_buf = pynq.allocate((N_SAMPLES,), dtype=sample_t, target=target)
mm2s_buf = pynq.allocate((SIZE_IN,), dtype=np.uint8, target=target)
s2mm_buf = pynq.allocate((SIZE_OUT,), dtype=np.uint8, target=target)

# np.random.seed(0)
# mm2s_buf[:] = np.random.randint(low=0, high=256, size=SIZE_IN, dtype=np.uint8)
# mm2s_buf.sync_to_device()

# --- Build input from patterns (repeat patterns to fill SIZE_IN) ---
pattern_file = Path("/home/m2_1/dat480_project_base/Project_resources/MINI_pattern_match_snort3_content.txt")

def parse_snort_pattern_line(line: str) -> bytes:
    """
    Parse one Snort-style content pattern line into bytes.
    Supports:
      - Pure ASCII lines: "GET /index.html"
      - Hex blocks like:  "ABC|0d 0a|DEF"
    """
    s = line.strip()
    if not s:
        return b""

    out = bytearray()
    i = 0
    while i < len(s):
        if s[i] == "|":
            j = s.find("|", i + 1)
            if j == -1:
                # unmatched '|', treat the rest as ASCII
                out.extend(s[i:].encode("utf-8", errors="ignore"))
                break
            hex_part = s[i+1:j].strip()
            # hex_part may contain spaces
            if hex_part:
                for tok in hex_part.split():
                    # allow tokens like "0A" or "0a"
                    out.append(int(tok, 16))
            i = j + 1
        else:
            out.append(ord(s[i]) & 0xFF)
            i += 1

    return bytes(out)

# 1) Read patterns
lines = pattern_file.read_text(encoding="utf-8", errors="ignore").splitlines()
patterns = []
for ln in lines:
    ln = ln.strip()
    if not ln:
        continue
    # optional: skip comment lines if exist
    if ln.startswith("#"):
        continue
    b = parse_snort_pattern_line(ln)
    if len(b) > 0:
        patterns.append(b)

if not patterns:
    raise RuntimeError("No valid patterns parsed from file.")

# 2) Build stream: pattern + separator, repeat until SIZE_IN
# Separator helps define boundaries; choose one your patterns likely don't rely on.
sep = b"\n"  # could use b"\x00" if you prefer
stream = bytearray()
k = 0
while len(stream) < SIZE_IN:
    p = patterns[k % len(patterns)]
    stream.extend(p)
    stream.extend(sep)
    k += 1

stream = stream[:SIZE_IN]

# 3) Copy into mm2s buffer
mm2s_buf[:] = np.frombuffer(stream, dtype=np.uint8)
mm2s_buf.sync_to_device()

print(f"Loaded {len(patterns)} patterns. Built input stream bytes={len(stream)} using repeat count={k}.")
print("First 64 input bytes:", mm2s_buf[:64].tolist())


# Optional: clear output/probe buffers
perf_buf[:] = 0
s2mm_buf[:] = 0
perf_buf.sync_to_device()
s2mm_buf.sync_to_device()

print("Buffers allocated and input synced to device.")


Loaded 256 patterns. Built input stream bytes=6464000 using repeat count=491471.
First 64 input bytes: [47, 98, 110, 98, 102, 111, 114, 109, 46, 99, 103, 105, 10, 47, 98, 98, 47, 105, 110, 100, 101, 120, 46, 112, 104, 112, 10, 47, 115, 101, 114, 118, 101, 114, 45, 115, 116, 97, 116, 117, 115, 10, 47, 110, 112, 104, 45, 101, 120, 112, 108, 111, 105, 116, 115, 99, 97, 110, 103, 101, 116, 46, 99, 103]
Buffers allocated and input synced to device.


In [6]:
# %% [markdown]
# ## Module 4 - Run pipeline (stop by perf_buf progress, not perf_w.done)

# %%
import time
import numpy as np

def peek_perf(prefix=""):
    try:
        perf_buf.sync_from_device()
        print(prefix, perf_buf[:5])
    except Exception as e:
        print(prefix, "perf_buf read failed:", e)

def count_samples_written():
    perf_buf.sync_from_device()
    return int(np.count_nonzero(perf_buf['cycles'] > 0))

# ---- Launch order: sink -> proj -> probe -> source ----
# Clear perf buffer so we can detect fresh writes
perf_buf[:] = 0
perf_buf.sync_to_device()

s2mm_w = ol.krnl_s2mm_0.start(s2mm_buf, SIZE_OUT, num_pkts)
proj_w = ol.krnl_proj_0.start(dest=0, num_packets=num_pkts)
perf_w = ol.stream_throughput_0.start(perf_buf, N_SAMPLES, N_RATE)
mm2s_w = ol.krnl_mm2s_0.start(mm2s_buf, SIZE_IN, 0)

print("Launched kernels.")
peek_perf(prefix="perf[0:5]=")

# ---- Wait until perf_buf has enough samples ----
timeout_s = 10
t0 = time.time()
last = -1

while True:
    n = count_samples_written()
    if n != last:
        last = n
        print(f"perf progress: {n}/{N_SAMPLES}")
        # optional: show the first few samples when progress changes
        peek_perf(prefix="perf[0:5]=")

    if n >= N_SAMPLES:
        print("All perf samples captured.")
        break

    if time.time() - t0 > timeout_s:
        print(f"WARN: timeout after {timeout_s}s; captured {n}/{N_SAMPLES}. Proceeding anyway.")
        break

    time.sleep(0.05)

# ---- Now it is safe to wait other kernels (they should finish with TLAST-driven s2mm) ----
def soft_wait(handle, name, timeout_s=10):
    t0 = time.time()
    while not handle.done and (time.time() - t0) < timeout_s:
        time.sleep(0.01)
    print(f"{name} done={handle.done}")

soft_wait(mm2s_w, "mm2s", 10)
soft_wait(proj_w, "proj", 10)
soft_wait(s2mm_w, "s2mm", 10)


Launched kernels.
perf[0:5]= [(0, 0, 0, 0) (0, 0, 0, 0) (0, 0, 0, 0) (0, 0, 0, 0) (0, 0, 0, 0)]
perf progress: 90/100
perf[0:5]= [(999, 576, 964, 0) (999, 816, 956, 0) (999, 704, 957, 0)
 (999, 480, 973, 0) (999, 624, 964, 0)]
perf progress: 100/100
perf[0:5]= [(999, 576, 964, 0) (999, 816, 956, 0) (999, 704, 957, 0)
 (999, 480, 973, 0) (999, 624, 964, 0)]
All perf samples captured.
mm2s done=False
proj done=False
s2mm done=False


In [7]:
# %% [markdown]
# ## Module 5 - Fetch samples and compute throughput

# %%
perf_buf.sync_from_device()

# Valid windows: bytes>0 is more meaningful for throughput
valid_mask = perf_buf['bytes'] > 0

print(f'Captured samples (bytes>0): {int(np.count_nonzero(valid_mask))}/{len(perf_buf)}')
print(perf_buf)

if not np.any(valid_mask):
    raise RuntimeError('No bytes counted in samples; stream_throughput saw no traffic')

cycles_total = int(perf_buf['cycles'][valid_mask].sum())
bytes_total = int(perf_buf['bytes'][valid_mask].sum())
ready_gaps = int(perf_buf['ready_not_valid'][valid_mask].sum())
backpressure = int(perf_buf['valid_not_ready'][valid_mask].sum())

print(
    f'\nTotals: cycles={cycles_total:,} bytes={bytes_total:,} '
    f'ready_not_valid={ready_gaps:,} valid_not_ready={backpressure:,}'
)

# Use fixed kernel clock unless you have a reliable readback
fclk_hz = 300_000_000

bytes_per_cycle = bytes_total / max(cycles_total, 1)
gbps = bytes_per_cycle * 8 * fclk_hz / 1e9

print(f'Bytes per cycle ≈ {bytes_per_cycle:.4f}')
print(f'Estimated throughput ≈ {gbps:.2f} Gbps @ fclk={fclk_hz/1e6:.1f} MHz')

# Per-sample throughput distribution
sample_bpc = perf_buf['bytes'][valid_mask] / np.maximum(perf_buf['cycles'][valid_mask], 1)
sample_gbps = sample_bpc * 8 * fclk_hz / 1e9
print(
    f'Per-sample throughput (Gbps) min/median/max: '
    f'{sample_gbps.min():.2f} / {np.median(sample_gbps):.2f} / {sample_gbps.max():.2f}'
)

# Optional: compare to theoretical scanner limit (if applicable)
if DCAM_BYTES_PER_CYCLE:
    theoretical_gbps = DCAM_BYTES_PER_CYCLE * 8 * fclk_hz / 1e9
    utilization = bytes_per_cycle / DCAM_BYTES_PER_CYCLE
    print(
        f'Kernel limit ({DCAM_BYTES_PER_CYCLE} B/cycle) ≈ {theoretical_gbps:.2f} Gbps; '
        f'utilization ≈ {utilization*100:.1f}%'
    )

print(f'Backpressure ratio (valid_not_ready / cycles): {backpressure / max(cycles_total,1):.3f}')
print(f'Ready-not-valid ratio: {ready_gaps / max(cycles_total,1):.3f}')


Captured samples (bytes>0): 100/100
[(999, 576, 964, 0) (999, 816, 956, 0) (999, 704, 957, 0)
 (999, 480, 973, 0) (999, 624, 964, 0) (999, 608, 962, 0)
 (999, 608, 964, 0) (999, 624, 961, 0) (999, 608, 962, 0)
 (999, 880, 954, 0) (999, 592, 962, 0) (999, 528, 967, 0)
 (999, 688, 961, 0) (999, 480, 969, 0) (999, 656, 958, 0)
 (999, 656, 959, 0) (999, 576, 966, 0) (999, 816, 956, 0)
 (999, 624, 961, 0) (999, 672, 961, 0) (999, 496, 969, 0)
 (999, 592, 962, 0) (999, 704, 958, 0) (999, 592, 965, 0)
 (999, 496, 968, 0) (999, 896, 950, 0) (999, 640, 963, 0)
 (999, 560, 967, 0) (999, 544, 967, 0) (999, 672, 960, 0)
 (999, 560, 965, 0) (999, 608, 963, 0) (999, 656, 962, 0)
 (999, 848, 954, 0) (999, 496, 969, 0) (999, 704, 963, 0)
 (999, 576, 964, 0) (999, 544, 965, 0) (999, 592, 963, 0)
 (999, 688, 959, 0) (999, 656, 961, 0) (999, 736, 960, 0)
 (999, 592, 965, 0) (999, 656, 961, 0) (999, 496, 968, 0)
 (999, 720, 957, 0) (999, 592, 963, 0) (999, 608, 961, 0)
 (999, 624, 962, 0) (999, 816, 957, 